In [23]:
import math
import random
from datetime import date, timedelta

#USER INPUT
INSTALL_DATE = date(2026, 1, 1)    #yyy/mm/dd
TARGET_DATE  = date(2071, 8, 17)

#KONSTANTA
A           = 1.0
ETA_REF     = 0.20
MU_T        = -0.0045
T_REF       = 25.0
NOCT        = 45.0
D_R         = 0.005
P_RATED     = 300.0
G_SC        = 1367.0
TAU_A       = 0.75
PHI         = math.radians(-6.8945)
L_DUST_MAX  = 0.15
LAMBDA      = 0.10
L_DOWN      = 0.02
GAMMA       = 0.025
N_STEPS     = 48   #buat integrasi


def system_age(current_date):
    """t = number of days since install / 365.25  (Appendix, symbol t)"""
    return (current_date - INSTALL_DATE).days / 365.25


def doy(d):
    return d.timetuple().tm_yday

def month(d):
    return d.month

#solar position
def declination(d):
    """
    Figure 2.8: δ(d) = 23.45 · sin(360/365 · (d - 81))
    d = day of year
    Returns radians.
    """
    return math.radians(23.45 * math.sin(math.radians((360.0 / 365.0) * (d - 81))))

def hour_angle(h):
    """
    Figure 2.9: ω(h) = 15(h - 12)
    h = hour of day [0–24]
    Returns radians.
    """
    return math.radians(15.0 * (h - 12.0))

def solar_elevation(h, d):
    """
    Figure 2.7: αs(h,d) = arcsin(sin φ sin δ + cos φ cos δ cos ω)
    Returns radians.
    """
    delta = declination(d)
    omega = hour_angle(h)
    return math.asin(
        math.sin(PHI) * math.sin(delta) +
        math.cos(PHI) * math.cos(delta) * math.cos(omega)
    )

#solar iradiance
def epsilon(target_date):
    """
    Appendix: ε(d) ~ N(0, 1)
    Seed is based on the date so the same date always gives the same ε,
    but different dates give different values.
    """
    seed = target_date.year * 10000 + target_date.month * 100 + target_date.day
    rng = random.Random(seed)
    # PAKE BOX MULLER
    u1 = rng.random()
    u2 = rng.random()
    return math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

def clearness_index(target_date):
    """
    Figure 2.10: KT(m,d) = max(0.1, min(1.0, 0.463 + 0.10·cos(2π(m-8)/12) + 0.08·ε(d)))
    """
    m = month(target_date)
    eps = epsilon(target_date)
    KT = 0.463 + 0.10 * math.cos(2.0 * math.pi * (m - 8.0) / 12.0) + 0.08 * eps
    return max(0.1, min(1.0, KT))

def diffuse_fraction(KT):
    """
    Figure 2.11: Erbs correlation
    fd(KT) = 1 - 0.09·KT                                        if KT ≤ 0.22
           = 0.9511 - 0.1604·KT + 4.388·KT² - 16.638·KT³ + 12.336·KT⁴  if 0.22 < KT ≤ 0.80
           = 0.165                                               if KT > 0.80
    """
    if KT <= 0.22:
        return 1.0 - 0.09 * KT
    elif KT <= 0.80:
        return (0.9511
                - 0.1604 * KT
                + 4.388  * KT**2
                - 16.638 * KT**3
                + 12.336 * KT**4)
    else:
        return 0.165

def G_total(h, target_date, KT):
    """
    Figure 2.6: G_total(h,d) = G_sc · τ_a · sin(αs(h,d)) · [KT(m,d) + fd(KT(m,d))]
    Returns 0 if sun is below horizon (αs ≤ 0).
    """
    alpha_s = solar_elevation(h, doy(target_date))
    if alpha_s <= 0:
        return 0.0
    fd = diffuse_fraction(KT)
    return G_SC * TAU_A * math.sin(alpha_s) * (KT + fd * KT)

#iradiasi harian
def H_daily(target_date, KT):
    """
    Figure 2.5: H(d) = ∫₀²⁴ G_total(h,d) · 1[αs(h,d) > 0] dh
    Numerically integrated using N_STEPS intervals.
    """
    dh = 24.0 / N_STEPS
    total = 0.0
    for i in range(N_STEPS):
        h = i * dh  # left-endpoint rule
        alpha_s = solar_elevation(h, doy(target_date))
        if alpha_s > 0:
            total += G_total(h, target_date, KT) * dh
    return total

#Tamb
def T_amb(h, target_date):
    """
    Figure 2.4: T_amb(h,d,t) = 23 + 6·sin(2π(h-6)/24) + γ·t
    t = system age in years from install date.
    """
    t = system_age(target_date)
    return 23.0 + 6.0 * math.sin(2.0 * math.pi * (h - 6.0) / 24.0) + GAMMA * t

#NOCT
def T_cell(h, target_date, KT):
    """
    Figure 2.3: T_C = T_amb + ((NOCT - 20) / 800) · G_total
    """
    G = G_total(h, target_date, KT)
    return T_amb(h, target_date) + ((NOCT - 20.0) / 800.0) * G

#efficiency
def eta(h, target_date, KT):
    """
    Figure 2.2: η(h,d,t) = η_ref · (1 + μT·(TC - T_ref)) · (1 - Dr)^t
    t = system age in years.
    """
    t  = system_age(target_date)
    Tc = T_cell(h, target_date, KT)
    return ETA_REF * (1.0 + MU_T * (Tc - T_REF)) * (1.0 - D_R)**t

#dust model
def p_rain(target_date):
    """
    Figure 2.14: p_rain(m) = 0.45 + 0.25·cos(2π(m-1)/12)
    """
    m = month(target_date)
    return 0.45 + 0.25 * math.cos(2.0 * math.pi * (m - 1.0) / 12.0)

def simulate_dust(n_days):
    """
    Figure 2.13: L_dust(d) = L_dust,max · (1 - e^(-λ·Δd))
    Δd resets to 0 when it rains (random < p_rain).
    Seed per day so rain events are reproducible.
    Returns list of L_dust for each day from install to target.
    """
    dust = []
    delta_d = 0
    for i in range(n_days):
        current = INSTALL_DATE + timedelta(days=i)
        # seed rain randomness by date
        seed = current.year * 10000 + current.month * 100 + current.day + 1
        rng  = random.Random(seed)
        if rng.random() < p_rain(current):
            delta_d = 0
        else:
            delta_d += 1
        dust.append(L_DUST_MAX * (1.0 - math.exp(-LAMBDA * delta_d)))
    return dust

#inverter loss model
def L_inv(h, target_date, KT):
    """
    Figure 2.15: L_inv(h,d) based on P/P_rated
        0.12  if P/P_rated < 0.10
        0.06  if 0.10 ≤ P/P_rated < 0.50
        0.04  if P/P_rated ≥ 0.50

    Figure 2.16: P(h,d,t) = G_total(h,d) · A · η(h,d,t)
    """
    G  = G_total(h, target_date, KT)
    et = eta(h, target_date, KT)
    P  = G * A * et
    ratio = P / P_RATED
    if ratio < 0.10:   return 0.12
    elif ratio < 0.50: return 0.06
    else:              return 0.04

#performance ratiio
def PR(h, target_date, KT, L_dust_val):
    """
    Figure 2.12: PR(h,d) = (1 - L_dust(d)) · (1 - L_inv(h,d)) · (1 - L_down)
    """
    return (1.0 - L_dust_val) * (1.0 - L_inv(h, target_date, KT)) * (1.0 - L_DOWN)

#E DAY
def compute_E_day(target_date, L_dust_val, KT, verbose=False):
    """
    Figure 2.1: E_day = A · η · H · PR
    Expanded form integrating over daylight hours.

    Parameters:
        target_date : date  — the day to compute
        L_dust_val  : float — dust loss for that day (from simulate_dust)
        KT          : float — clearness index for that day
        verbose     : bool  — print hourly breakdown

    Returns:
        E_day in Wh
    """
    dh = 24.0 / N_STEPS

    if verbose:
        print(f"\n  {'h':>5} {'as(deg)':>9} {'G(W/m2)':>9} "
              f"{'Tc(C)':>7} {'eta(%)':>7} {'PR':>6} {'dE(Wh)':>8}")
        print("  " + "-" * 57)

    E = 0.0
    for i in range(N_STEPS):
        h       = i * dh
        alpha_s = solar_elevation(h, doy(target_date))
        G       = G_total(h, target_date, KT)
        if alpha_s <= 0 or G <= 0:
            continue

        et  = eta(h, target_date, KT)
        pr  = PR(h, target_date, KT, L_dust_val)
        dE  = A * et * G * dh * pr
        E  += dE

        if verbose:
            Tc = T_cell(h, target_date, KT)
            print(f"  {h:>5.2f} {math.degrees(alpha_s):>9.3f} {G:>9.2f} "
                  f"{Tc:>7.2f} {et*100:>7.3f} {pr:>6.4f} {dE:>8.4f}")

    if verbose:
        print("  " + "-" * 57)

    return E

#E AVG
def compute_E_avg(verbose=False):
    """
    Figure 2.2: E_avg = (1/N) · Σ_{i=1}^{N} E_day(i)
    N = (TARGET_DATE - INSTALL_DATE).days

    Returns:
        E_avg   : float — mean Wh/day over the period
        daily_E : list  — E_day for each day in the period
    """
    N = (TARGET_DATE - INSTALL_DATE).days
    if N <= 0:
        raise ValueError("TARGET_DATE must be after INSTALL_DATE")

    dust = simulate_dust(N)

    daily_E = []
    for i in range(N):
        d   = INSTALL_DATE + timedelta(days=i)
        KT  = clearness_index(d)
        E   = compute_E_day(d, dust[i], KT)
        daily_E.append(E)
        if verbose and (i + 1) % 30 == 0:
            print(f"    day {i+1:>3}/{N} — E_day = {E:.2f} Wh")

    E_avg = sum(daily_E) / N
    return E_avg, daily_E

#main runtime
if __name__ == "__main__":
    N = (TARGET_DATE - INSTALL_DATE).days
    if N <= 0:
        raise ValueError("TARGET_DATE must be after INSTALL_DATE")

    print("=" * 55)
    print("  PV Simulation — Bandung, Indonesia")
    print("  Crisbar, MCF ITB 2026")
    print("=" * 55)
    print(f"  Install date  (d_ref) : {INSTALL_DATE.strftime('%d %b %Y')}")
    print(f"  Target date           : {TARGET_DATE.strftime('%d %b %Y')}")
    print(f"  N (days)              : {N}")
    print(f"  System age on target  : {N/365.25:.4f} years\n")

    #e avg output
    print("  Computing E_avg...")
    E_avg_val, daily_E = compute_E_avg(verbose=False)

    #e day output
    dust_all      = simulate_dust(N + 1)
    L_dust_target = dust_all[N]
    KT_target     = clearness_index(TARGET_DATE)

    print(f"\n  Hourly breakdown for {TARGET_DATE.strftime('%d %b %Y')}:")
    E_day_val = compute_E_day(TARGET_DATE, L_dust_target, KT_target, verbose=True)

    # summary
    print(f"\n{'=' * 55}")
    print(f"  RESULTS")
    print(f"{'=' * 55}")
    print(f"  KT on {TARGET_DATE.strftime('%d %b %Y')}       : {KT_target:.4f}")
    print(f"  Dust loss on target day  : {L_dust_target*100:.4f}%")
    print(f"\n  E_day  ({TARGET_DATE.strftime('%d %b %Y')})  : {E_day_val:.4f} Wh")
    print(f"                            ({E_day_val/1000:.6f} kWh)")
    print(f"\n  E_avg  ({INSTALL_DATE.strftime('%d %b %Y')} to {TARGET_DATE.strftime('%d %b %Y')})")
    print(f"         N = {N} days        : {E_avg_val:.4f} Wh/day")
    print(f"                            ({E_avg_val/1000:.6f} kWh/day)")
    print(f"\n  E_day vs E_avg           : {((E_day_val - E_avg_val)/E_avg_val)*100:+.2f}%")
    print(f"{'=' * 55}")

  PV Simulation — Bandung, Indonesia
  Crisbar, MCF ITB 2026
  Install date  (d_ref) : 01 Jan 2026
  Target date           : 17 Aug 2071
  N (days)              : 16664
  System age on target  : 45.6235 years

  Computing E_avg...

  Hourly breakdown for 17 Aug 2071:

      h   as(deg)   G(W/m2)   Tc(C)  eta(%)     PR   dE(Wh)
  ---------------------------------------------------------
   6.50     5.678     84.06   27.55  15.729 0.7683   5.0789
   7.00    12.884    189.43   31.61  15.438 0.7683  11.2343
   7.50    20.044    291.17   35.54  15.157 0.8207  18.1096
   8.00    27.140    387.53   39.25  14.891 0.8207  23.6797
   8.50    34.147    476.86   42.70  14.644 0.8207  28.6558
   9.00    41.027    557.64   45.81  14.422 0.8207  32.9996
   9.50    47.714    628.48   48.54  14.226 0.8207  36.6874
  10.00    54.102    688.17   50.84  14.061 0.8207  39.7065
  10.50    59.997    735.69   52.67  13.930 0.8207  42.0523
  11.00    65.046    770.23   54.01  13.835 0.8207  43.7250
  11.50    